#### **Data analysis for bacteria gene annotation - version: TIS generation**

In [26]:
#pip install biopython
#!pip install orfipy
#!pip install intervaltree
#!pip install line-profiler
#!pip install memory-profiler
#!pip install filelock

In [27]:
import pandas as pd
import os

#### **Verified genome summary**

In [28]:
verified_sample_path = "../data/verified-genome/raw"

data = []
for folder in os.listdir(verified_sample_path):
    data.append({
        "bacteria": folder, 
        "genome_file_path": os.path.join(verified_sample_path, folder, 'sequence.fasta'), 
        "verified_annotation_path": os.path.join(verified_sample_path, folder, 'verified.gff'),
        "ncbi_annotation_path": os.path.join(verified_sample_path, folder, 'ncbi.gff'),
    })

bacteria_summary = pd.DataFrame(data)
bacteria_summary.head()

,bacteria,genome_file_path,verified_annotation_path,ncbi_annotation_path
0,Escherichia_coli_K_12_substr__MG1655_uid57779,../data/verified-genome/raw/Escherichia_coli_K...,../data/verified-genome/raw/Escherichia_coli_K...,../data/verified-genome/raw/Escherichia_coli_K...
1,Mycobacterium_tuberculosis_H37Rv_uid57777,../data/verified-genome/raw/Mycobacterium_tube...,../data/verified-genome/raw/Mycobacterium_tube...,../data/verified-genome/raw/Mycobacterium_tube...
2,Roseobacter_denitrificans_Och114,../data/verified-genome/raw/Roseobacter_denitr...,../data/verified-genome/raw/Roseobacter_denitr...,../data/verified-genome/raw/Roseobacter_denitr...
3,Natronomonas_pharaonis_DSM_2160_uid58435,../data/verified-genome/raw/Natronomonas_phara...,../data/verified-genome/raw/Natronomonas_phara...,../data/verified-genome/raw/Natronomonas_phara...
4,Halobacterium_salinarum_R1_uid61571,../data/verified-genome/raw/Halobacterium_sali...,../data/verified-genome/raw/Halobacterium_sali...,../data/verified-genome/raw/Halobacterium_sali...


#### **Data Extracting & Labeling**

In [29]:
%load_ext cython
%load_ext line_profiler
%load_ext memory_profiler

The cython extension is already loaded. To reload it, use:
  %reload_ext cython
The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
The memory_profiler extension is already loaded. To reload it, use:
  %reload_ext memory_profiler


In [30]:
import subprocess, gc, warnings, os, re, json, gzip, random, logging
import pandas as pd
import numpy as np
from tqdm import tqdm
from Bio import SeqIO
from itertools import groupby
import multiprocessing as mp
import concurrent.futures
import orfipy_core as oc
from sklearn.utils import shuffle
from filelock import FileLock

logging.basicConfig(
    filename='logger_tis.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

warnings.simplefilter(action='ignore', category=FutureWarning)
gc.collect()

776

In [31]:
#%%cython -a
def extract_dna(sequence, start_pos=None, end_pos=None):
    if start_pos is not None and end_pos is not None:
        return sequence[start_pos:end_pos]
    return sequence

In [32]:
def parse_orfs(input_data):
    orfs_pos, orfs_neg = [], []
    orf_data = input_data.strip().split('>')[1:]
    
    for entry in orf_data:
        lines = entry.splitlines()
        header = lines[0]
        sequence = ''.join(lines[1:])
        
        # extract start, end, strand, frame, length, start codon, end codon from the header using regex
        match = re.search(r'(\d+)-(\d+)\]\((\+|-)\) type:complete length:(\d+) frame:(-?\d+) start:(\w{3}) stop:(\w{3})', header)
        if match:
            start = int(match.group(1))
            end = int(match.group(2))
            strand = match.group(3)
            length = int(match.group(4))
            frame = int(match.group(5))
            start_codon = match.group(6)
            end_codon = match.group(7)

            if strand == "+":
                orfs_pos.append({
                    'start': start,
                    'end': end,
                    'strand': strand,
                    'length': length,
                    'frame': frame,
                    'start_codon': start_codon,
                    'end_codon': end_codon,
                    'sequence': sequence
                })
            elif strand == "-":
                orfs_neg.append({
                    'start': start,
                    'end': end,
                    'strand': strand,
                    'length': length,
                    'frame': frame,
                    'start_codon': start_codon,
                    'end_codon': end_codon,
                    'sequence': sequence
                })
    
    return orfs_pos, orfs_neg

In [33]:
#%%cython -a
def cds_annotator(cds_df, orfs_dict, strand, max_seq_len=510):
    results = []
    if strand == "+":
        positive_orfs   = orfs_dict
        for index, cds_row in cds_df.iterrows():
            seqid   = cds_row["seqid"]
            cds_end = cds_row["end"]
            matching_orfs  = [orf for orf in positive_orfs if orf['end'] == cds_end]
            positive_orfs  = [orf for orf in positive_orfs if orf['end'] != cds_end]
            if matching_orfs:
                longest_orf = max(matching_orfs, key=lambda d: d['length'])
                results.append({"sequence": longest_orf["sequence"][-max_seq_len:], "label": 1})
        # remaining orfs
        grouped = groupby(sorted(positive_orfs, key=lambda x: x['end']), key=lambda x: x['end'])
        for end_value, group in grouped:
            orf = max(group, key=lambda x: x['length'])
            if orf['length'] > 100:
                results.append({"sequence": orf["sequence"][-max_seq_len:], "label": 0})
    elif strand == "-":
        negative_orfs   = orfs_dict
        for index, cds_row in cds_df.iterrows():
            seqid   = cds_row["seqid"]
            cds_start = cds_row["start"]
            matching_orfs  = [orf for orf in negative_orfs if orf['start'] == cds_start]
            negative_orfs  = [orf for orf in negative_orfs if orf['start'] != cds_start]
            if matching_orfs:
                longest_orf = max(matching_orfs, key=lambda d: d['length'])
                results.append({"sequence": longest_orf["sequence"][:max_seq_len], "label": 1})
        # remaining orfs
        grouped = groupby(sorted(negative_orfs, key=lambda x: x['start']), key=lambda x: x['start'])
        for start_value, group in grouped:
            orf = max(group, key=lambda x: x['length'])
            if orf['length'] > 100:
                results.append({"sequence": orf["sequence"][:max_seq_len], "label": 0})
    return results


#%%cython -a
def tis_annotator(dna_seq, cds_df, orfs_dict, strand, tis_window=30):
    results = []
    if strand == "+":
        positive_orfs   = orfs_dict
        group = 0
        for index, cds_row in cds_df.iterrows():
            group += 1
            seqid   = cds_row["seqid"]
            cds_end = cds_row["end"]
            cds_start = cds_row["start"]
            matching_orfs  = [orf for orf in positive_orfs if orf['end'] == cds_end]
            if matching_orfs:
                for orf in matching_orfs:
                    if orf["start"] == cds_start:
                        results.append({"sequence": extract_dna(dna_seq, orf["start"]-tis_window, orf["start"]+tis_window), "label": 1,
                                        "orf_group": "P"+str(group)})
                    else:
                        # should we append all non-tis sequence
                        results.append({"sequence": extract_dna(dna_seq, orf["start"]-tis_window, orf["start"]+tis_window), "label": 0,
                                        "orf_group": "P"+str(group)})
    elif strand == "-":
        negative_orfs   = orfs_dict
        group = 0
        for index, cds_row in cds_df.iterrows():
            group += 1
            seqid   = cds_row["seqid"]
            cds_start = cds_row["start"]
            cds_end   = cds_row["end"]
            matching_orfs  = [orf for orf in negative_orfs if orf['start'] == cds_start]
            #print("<DEBUG>: ", len(matching_orfs))
            if matching_orfs:
                for orf in matching_orfs:
                    if orf["end"] == cds_end:
                        #results.append({"sequence": extract_dna(dna_seq, orf["end"]-tis_window, orf["end"]+tis_window), "label": 1})
                        #print({"sequence": extract_dna(dna_seq, orf["end"]-tis_window, orf["end"]+tis_window), "label": 1})
                        #print({"sequence": extract_dna(dna_seq, -(orf["end"]+tis_window), -(orf["end"]-tis_window)), "label": -1})
                        # fix bug: idx extraction wrong
                        results.append({"sequence": extract_dna(dna_seq, -(orf["end"]+tis_window), -(orf["end"]-tis_window)), "label": 1, 
                                        "orf_group": "N"+str(group)})
                    else:
                        #results.append({"sequence": extract_dna(dna_seq, orf["end"]-tis_window, orf["end"]+tis_window), "label": 0})
                        #print({"sequence": extract_dna(dna_seq, orf["end"]-tis_window, orf["end"]+tis_window), "label": 0})
                        #print({"sequence": extract_dna(dna_seq, -(orf["end"]+tis_window), -(orf["end"]-tis_window)), "label": -0})
                        # fix bug: idx extraction wrong
                        results.append({"sequence": extract_dna(dna_seq, -(orf["end"]+tis_window), -(orf["end"]-tis_window)), "label": 0, 
                                        "orf_group": "N"+str(group)})
                        #break
            #print("== STOP ==")
            #break
    return results

In [34]:
def data_balancer(df, id_):
    if df.shape[0] == 0:
        logging.warning(f'Empty dataset for bacteria: {id_}')
        return df
        
    positive_class = df[df['label'] == 1]
    negative_class = df[df['label'] == 0].copy()
    positive_lengths = positive_class['sequence'].apply(len)
    positive_length_distribution = positive_lengths.tolist()

    negative_class_balanced = []
    for length in positive_length_distribution:
        possible_negatives = negative_class[negative_class['sequence'].apply(len) == length]
        if not possible_negatives.empty:
            selected_sample = possible_negatives.sample(n=1, random_state=42)
            negative_class_balanced.append(selected_sample)
            
            # deL selected sample from negative_class to prevent duplicates
            negative_class = negative_class.drop(selected_sample.index)

    if negative_class_balanced:
        negative_class_balanced = pd.concat(negative_class_balanced, ignore_index=True)
    else:
        print("Warning: No matching negative samples found. Returning only positive class data.")
        logging.warning(f'No matching negative samples found. Returning only positive class data.: {id_}')
        negative_class_balanced = negative_class # pd.DataFrame(columns=df.columns)  # Empty DataFrame
        
    # check balance
    num_positives = len(positive_class)
    num_negatives = len(negative_class_balanced)

    if num_negatives < num_positives:
        missing_negatives = num_positives - num_negatives
        if len(negative_class) > 0:
            additional_negatives = negative_class.sample(n=min(missing_negatives, len(negative_class)), random_state=42)
            negative_class_balanced = pd.concat([negative_class_balanced, additional_negatives], ignore_index=True)
    
    # shuffle
    balanced_df = pd.concat([positive_class, negative_class_balanced], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df

In [35]:
etl_folder     = "../data/verified-genome2/"
json_path      = "../data/verified-genome2/cds/data_summary.json"
json_path2      = "../data/verified-genome2/tis/data_summary.json"
os.makedirs(etl_folder+"cds/database/", exist_ok=True)
os.makedirs(etl_folder+"tis/database/", exist_ok=True)
os.makedirs(etl_folder+"cds/balanced/", exist_ok=True)
os.makedirs(etl_folder+"tis/balanced/", exist_ok=True)
if not os.path.exists(json_path):
    with open(json_path, 'w') as file:
        json.dump({}, file)
if not os.path.exists(json_path2):
    with open(json_path2, 'w') as file:
        json.dump({}, file)

def extractor(row_index, row):
    # init
    debug = False
    folder_name = row['bacteria']
    gff_path = row['verified_annotation_path']
    fna_path = row['genome_file_path']
    
    # cds
    columns = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
    with open(gff_path, 'rt') as f:
        lines = [line.strip() for line in f if not line.startswith('#')]
    gff_data = [line.split('\t') for line in lines]
    gff_df = pd.DataFrame(gff_data, columns=columns)
    gff_df["start"] = pd.to_numeric(gff_df["start"])
    gff_df["start"] = gff_df["start"] - 1
    gff_df["end"]   = pd.to_numeric(gff_df["end"])
    gff_df["end"]   = gff_df["end"]
    
    cds_df          = gff_df[gff_df["type"] == "CDS"]
    if debug:
        cds_df.to_csv("temp/cds.csv")
    cds_pos_df = cds_df[cds_df["strand"] == "+"]
    cds_neg_df = cds_df[cds_df["strand"] == "-"]

    # fna
    if debug:
        with gzip.open(fna_path, 'rb') as genome_file:
            with open("temp/genome.fna", 'wb') as file:
                file.write(genome_file.read())
    with open(fna_path, 'rt') as f:
        fna_data = list(SeqIO.parse(f, "fasta"))

    # orf 
    df_list = []
    for record in fna_data:
        seq      = str(record.seq)#[:1000000]
        seq_rc   = str(record.seq.reverse_complement())#[:1000000]
        bacteria = record.description.split(",")[0]
        orfs     = oc.start_search(seq,                                                #seq
                                seq_rc,                                                #seq_rc
                                bacteria,                                              #seqname
                                10,                                                    #minlen
                                10000000,                                              #maxlen
                                'b',                                                   #strand
                                ['TTG', 'CTG', 'ATG', 'GTG'],                          #starts
                                ['TAA', 'TAG', 'TGA'],                                 #stops
                                '1',                                                   #table
                                True,                                                  #include_stop
                                False,                                                 #partial3
                                False,                                                 #partial5
                                False,                                                 #between_stops
                                True,                                                  #nested
                                [False,False,True,False,False]                         #[bed12, bed, dna, rna, pep]    
                )
        if debug:
            pass
        with open('temp/orfs_dna.fna', 'w') as file:
            file.write(orfs[2])
        orfs_pos_dict, orfs_neg_dict = parse_orfs(orfs[2])
        print("orf+: ", len(orfs_pos_dict))
        print("orf-: ", len(orfs_neg_dict))

        # processing
        exp = "tis"
        if exp == "cds":
            df1 = cds_annotator(cds_pos_df, orfs_pos_dict, "+")
            df2 = cds_annotator(cds_neg_df, orfs_neg_dict, "-")
        elif exp == "tis":
            df1 = tis_annotator(seq, cds_pos_df, orfs_pos_dict, "+")
            df2 = tis_annotator(seq_rc, cds_neg_df, orfs_neg_dict, "-")
        df_list.extend(df1)
        df_list.extend(df2)

    # save df & update data_summary
    df = pd.DataFrame(df_list)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    df_file_path = etl_folder+f"{exp}/database/bacteria_{row_index+1}.csv"
    df.to_csv(df_file_path, index=False)
    df_balanced = data_balancer(df, folder_name+"__"+str(row_index))
    df_balanced_file_path = etl_folder+f"{exp}/balanced/bacteria_{row_index+1}.csv"
    df_balanced.to_csv(df_balanced_file_path, index=False)

    # only one process writes at a time
    json_path_ = json_path if exp == "cds" else json_path2
    with FileLock(json_path_ + ".lock"):  
        with open(json_path_, 'r') as file:
            data_summary = json.load(file)
        data_summary[folder_name+f".{row_index}"] = {"bacteria": row["bacteria"], "path": df_file_path} #PATCH: (+.int) : same folder name
        with open(json_path_, 'w') as file:
            json.dump(data_summary, file, indent=4)

# -----------------------------------------------------------------------------------
annotated_summary_ = bacteria_summary.iloc[:]
for row_index, row in tqdm(annotated_summary_.iterrows(), desc="Parsing seq: "):
    extractor(row_index, row)

Parsing seq: : 0it [00:00, ?it/s]

orf+:  293245
orf-:  296428


Parsing seq: : 1it [00:25, 25.94s/it]

orf+:  264786
orf-:  264495


Parsing seq: : 2it [00:49, 24.46s/it]

...

Parsing seq: : 3it [01:07, 21.59s/it]

orf+:  117480
orf-:  117957
orf+:  6909
orf-:  7182
orf+:  1280
orf-:  1132


Parsing seq: : 4it [01:12, 14.99s/it]

orf+:  95918
orf-:  96667
orf+:  7912
orf-:  7293
orf+:  2276
orf-:  2057
orf+:  15059
orf-:  14795
orf+:  10173
orf-:  9551


Parsing seq: : 5it [01:20, 16.18s/it]


In [36]:
bacteria_summary.iloc[0]['verified_annotation_path']

'../data/verified-genome/raw/Escherichia_coli_K_12_substr__MG1655_uid57779/verified.gff'

In [37]:
os.cpu_count() 

56

In [38]:
import json, os

def train_test_file_split(train_range, test_range, dev_range, save_path, is_list=False):
    # internal utils
    def generate_file_list(start, end):
        return [f"bacteria_{i}.csv" for i in range(start, end+1)]
    
    # generate
    if not is_list:
        train_files = generate_file_list(*train_range)
        test_files  = generate_file_list(*test_range)
        dev_files   = generate_file_list(*dev_range)
    else:
        train_files = train_range
        test_files  = test_range
        dev_files   = dev_range

    data = {
        "train": train_files,
        "test":  test_files,
        "dev":   dev_files,
    }

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, "w") as json_file:
        json.dump(data, json_file, indent=4)
    
    return True
    


In [39]:
train_test_file_split([1,3], [4, 4], [5, 5], "../data/verified-genome/cds/data_index.json")
train_test_file_split([1,3], [4, 4], [5, 5], "../data/verified-genome/tis/data_index.json")

True

In [40]:
# End